In [1]:
%pip install yfinance tensorflow

import yfinance as yf
import pandas as pd
import numpy as np
import random
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
tickers = ["QQQ", "BB", "M", "CMG"]
prices = {}

for ticker_symbol in tickers:
    ticker = yf.Ticker(ticker_symbol)
    prices[ticker_symbol] = ticker.history(start="2009-02-14", end="2020-06-11")
    print(f"\n{ticker_symbol}:")
    print(prices[ticker_symbol])


QQQ:
                                 Open        High         Low       Close  \
Date                                                                        
2009-02-17 00:00:00-05:00   25.455397   25.610771   25.084226   25.222336   
2009-02-18 00:00:00-05:00   25.386341   25.610770   24.920221   25.239599   
2009-02-19 00:00:00-05:00   25.394971   25.558977   24.790739   24.851164   
2009-02-20 00:00:00-05:00   24.574945   25.161913   24.505890   24.920221   
2009-02-23 00:00:00-05:00   25.058332   25.075595   23.936187   24.048403   
...                               ...         ...         ...         ...   
2020-06-04 00:00:00-04:00  228.241103  229.651124  225.681796  226.985596   
2020-06-05 00:00:00-04:00  228.134896  232.075252  227.565094  231.486130   
2020-06-08 00:00:00-04:00  231.341242  233.407996  229.767043  233.282455   
2020-06-09 00:00:00-04:00  232.422955  235.822466  232.239456  234.972595   
2020-06-10 00:00:00-04:00  236.662696  239.337891  236.141170  237.792

In [3]:
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    # correct next-day direction target
    price["Score"] = (price["Close"].shift(-1) > price["Close"]).astype(int)

    # lagged features
    price["Change"]      = price["Close"].shift(1) - price["Open"].shift(1)
    price["%Change"]     = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
    price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
    price["YestScore"]   = price["Score"].shift(1)
    price["5DayMean"]    = price["Close"].rolling(5).mean().shift(1)
    price["5DayVoli"]    = price["Close"].rolling(5).std().shift(1)
    price["5DayPerf"]    = price["Score"].rolling(5).sum().shift(1)

    price.dropna(inplace=True)
    prices[ticker_symbol] = price

In [4]:
features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]

window_size = 5
all_results = {}

def make_windows(feature_data, target_data, dates, window_size):
    X = []
    y = []
    d = []

    for i in range(window_size, len(feature_data)):
        X.append(feature_data[i-window_size:i])
        y.append(target_data[i])
        d.append(dates[i])

    return np.array(X), np.array(y), np.array(d)

In [5]:
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    priceTrain = price[price.index <= "2018-06-11"].copy()
    priceTest  = price[price.index > "2018-06-11"].copy()

    scaler = StandardScaler()

    xTrain_raw = priceTrain[features]
    xTest_raw  = priceTest[features]

    xTrain_scaled = scaler.fit_transform(xTrain_raw)
    xTest_scaled  = scaler.transform(xTest_raw)

    yTrain = priceTrain["Score"].values
    yTest  = priceTest["Score"].values

    train_dates = priceTrain.index
    test_dates  = priceTest.index

    X_train, y_train, d_train = make_windows(xTrain_scaled, yTrain, train_dates, window_size)

    # For test windows, prepend the last training rows so the first test point has enough history
    combined_X = np.vstack([xTrain_scaled[-window_size:], xTest_scaled])
    combined_y = np.concatenate([yTrain[-window_size:], yTest])
    combined_d = np.concatenate([train_dates[-window_size:], test_dates])

    X_test, y_test, d_test = make_windows(combined_X, combined_y, combined_d, window_size)

    model = Sequential([
        LSTM(32, input_shape=(window_size, len(features))),
        Dropout(0.2),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    probs = model.predict(X_test, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)

    results = pd.DataFrame({
        "Date": d_test,
        "Score": y_test,
        "Prediction": preds,
        "Probability": probs
    })

    results["Open"] = price.loc[results["Date"], "Open"].values
    results["Close"] = price.loc[results["Date"], "Close"].values

    all_results[ticker_symbol] = results

    print(f"\nFinished {ticker_symbol}")
    print(results.head())

C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished QQQ
                       Date  Score  Prediction  Probability        Open  \
0 2018-06-12 00:00:00-04:00      0           1     0.534764  166.282383   
1 2018-06-13 00:00:00-04:00      1           1     0.527691  167.184234   
2 2018-06-14 00:00:00-04:00      0           0     0.478630  167.687446   
3 2018-06-15 00:00:00-04:00      0           1     0.520494  167.972243   
4 2018-06-18 00:00:00-04:00      0           0     0.464255  166.961787   

        Close  
0  166.927963  
1  166.918411  
2  168.608337  
3  168.019714  
4  167.922699  


C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished BB
                       Date  Score  Prediction  Probability   Open  Close
0 2018-06-12 00:00:00-04:00      0           0     0.437614  12.10  12.33
1 2018-06-13 00:00:00-04:00      0           0     0.449937  12.40  12.33
2 2018-06-14 00:00:00-04:00      1           0     0.474503  12.37  12.22
3 2018-06-15 00:00:00-04:00      0           0     0.480519  12.17  12.31
4 2018-06-18 00:00:00-04:00      0           0     0.476711  12.19  12.20


C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished M
                       Date  Score  Prediction  Probability       Open  \
0 2018-06-12 00:00:00-04:00      0           1     0.521902  28.557115   
1 2018-06-13 00:00:00-04:00      0           1     0.539562  28.542838   
2 2018-06-14 00:00:00-04:00      1           1     0.583878  27.615480   
3 2018-06-15 00:00:00-04:00      1           1     0.559516  26.923288   
4 2018-06-18 00:00:00-04:00      1           1     0.576525  27.413593   

       Close  
0  28.535698  
1  27.393414  
2  27.081919  
3  27.593847  
4  27.925524  


C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished CMG
                       Date  Score  Prediction  Probability    Open   Close
0 2018-06-12 00:00:00-04:00      0           0     0.482957  9.3288  9.3260
1 2018-06-13 00:00:00-04:00      1           1     0.519943  9.3382  9.1460
2 2018-06-14 00:00:00-04:00      1           1     0.531999  9.1864  9.2088
3 2018-06-15 00:00:00-04:00      1           0     0.490664  9.1976  9.2402
4 2018-06-18 00:00:00-04:00      1           0     0.473137  9.1804  9.3692


In [6]:
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    accuracy  = accuracy_score(score, pred)
    precision = precision_score(score, pred, zero_division=0)
    recall    = recall_score(score, pred, zero_division=0)
    f1        = f1_score(score, pred, zero_division=0)
    mcc       = matthews_corrcoef(score, pred)

    print(f"\n{ticker_symbol}:")
    print("Mean:", results["Score"].mean())
    print("Accuracy: ", accuracy)
    print("Precision:", precision)
    print("Recall:   ", recall)
    print("F1:       ", f1)
    print("MCC:      ", mcc)


QQQ:
Mean: 0.5765407554671969
Accuracy:  0.5467196819085487
Precision: 0.5824468085106383
Recall:    0.7551724137931034
F1:        0.6576576576576577
MCC:       0.020566888305344085

BB:
Mean: 0.4691848906560636
Accuracy:  0.5228628230616302
Precision: 0.4523809523809524
Recall:    0.08050847457627118
F1:        0.1366906474820144
MCC:       -0.010163460719040104

M:
Mean: 0.48111332007952284
Accuracy:  0.4691848906560636
Precision: 0.4605678233438486
Recall:    0.6033057851239669
F1:        0.5223613595706619
MCC:       -0.053682144062639256

CMG:
Mean: 0.5546719681908548
Accuracy:  0.49502982107355864
Precision: 0.5683060109289617
Recall:    0.3727598566308244
F1:        0.45021645021645024
MCC:       0.0207451739601547


In [7]:
for ticker_symbol, results in all_results.items():
    totalIn = 0
    totalOut = 0
    totalInvested = 0
    totalStoodOut = 0
    alltradein = 0
    alltradeout = 0
    perftradein = 0
    perftradeout = 0

    for date, pred, score in zip(results["Date"], results["Prediction"], results["Score"]):
        current_close = prices[ticker_symbol].loc[date, "Close"]

        next_idx = prices[ticker_symbol].index.get_loc(date) + 1
        if next_idx >= len(prices[ticker_symbol]):
            continue

        next_close = prices[ticker_symbol].iloc[next_idx]["Close"]
        ret_percent = ((next_close - current_close) / current_close) * 100

        if pred == 1:
            totalIn += 100
            totalInvested += 1
            totalOut += 100 + ret_percent
        else:
            totalStoodOut += 1

        alltradein += 100
        alltradeout += 100 + ret_percent

        if score == 1:
            perftradein += 100
            perftradeout += 100 + ret_percent

    print(f"\n{ticker_symbol}:")
    print(f"  Model:    Invested ${totalIn:.0f}, Returned ${totalOut:.2f}, Profit ${totalOut - totalIn:.2f} ({totalInvested} trades, {totalStoodOut} skipped)")
    print(f"  Buy&Hold: Invested ${alltradein:.0f}, Returned ${alltradeout:.2f}, Profit ${alltradeout - alltradein:.2f}")
    print(f"  Perfect:  Invested ${perftradein:.0f}, Returned ${perftradeout:.2f}, Profit ${perftradeout - perftradein:.2f}")


QQQ:
  Model:    Invested $37500, Returned $37533.59, Profit $33.59 (375 trades, 127 skipped)
  Buy&Hold: Invested $50200, Returned $50243.01, Profit $43.01
  Perfect:  Invested $29000, Returned $29296.20, Profit $296.20

BB:
  Model:    Invested $4200, Returned $4178.07, Profit $-21.93 (42 trades, 460 skipped)
  Buy&Hold: Invested $50200, Returned $50150.43, Profit $-49.57
  Perfect:  Invested $23600, Returned $24110.42, Profit $510.42

M:
  Model:    Invested $31600, Returned $31507.31, Profit $-92.69 (316 trades, 186 skipped)
  Buy&Hold: Invested $50200, Returned $50095.15, Profit $-104.85
  Perfect:  Invested $24200, Returned $24783.08, Profit $583.08

CMG:
  Model:    Invested $18300, Returned $18394.78, Profit $94.78 (183 trades, 319 skipped)
  Buy&Hold: Invested $50200, Returned $50295.25, Profit $95.25
  Perfect:  Invested $27900, Returned $28349.91, Profit $449.91
